In [0]:
%pip install python-Levenshtein biotite gemmi py2Dmol

In [0]:
%restart_python

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

### Combine Metrics across all Protein-Hunter Runs


In [0]:
import os
import pandas as pd
def merge_metrics(overarching_folder_path: str) -> pd.DataFrame:
    """ Merges the metrics from the 2 different metric assessment files into a single dataframe.
            - Apo Boltz Metrics
            - Holo AF2 & Boltz Metrics
            - Physics Metrics
    Args:
        overarching_folder_path (str): Path to the folder containing the metrics files.
        
    Returns:
        pd.DataFrame: Merged metrics dataframe
    """
    path_final_metrics = os.path.join(overarching_folder_path, "final_metrics")
    for filename in os.listdir(path_final_metrics):
        
        if "physics" in filename:
            path_physics_metrics = os.path.join(path_final_metrics, filename)
            df_physics_metrics = pd.read_csv(path_physics_metrics, index_col= 0)
            df_physics_final = df_physics_metrics.copy()
            df_physics_final['design_name'] = df_physics_final['pdb_filename'].str.split('_model_').str[0]
        
        elif "apo" in filename:
            path_apo_metrics = os.path.join(path_final_metrics, filename)
            df_apo_metrics = pd.read_csv(path_apo_metrics)
            agg_funcs = ['mean', 'median', 'std']
            df_apo_agg = df_apo_metrics.groupby('design_name').agg({
                                                            "ptm" : agg_funcs,
                                                            "complex_plddt" : agg_funcs,
                                                            })
            df_apo_agg.reset_index(inplace = True)
            df_apo_agg.columns = [("_".join(col)).strip("_") for col in df_apo_agg.columns]
            df_apo_model0 = df_apo_metrics[df_apo_metrics['model_id'] == 0].copy()
            df_apo_agg = pd.merge(df_apo_agg, df_apo_model0[['design_name', 'path_boltz_apo_structure']], on = 'design_name', how = 'left')
            df_apo_final = df_apo_agg.rename(columns = {'ptm_mean': 'ptm_apo', 'complex_plddt_mean': 'plddt_apo'})
        
        elif "holo" in filename:
            path_holo_metrics = os.path.join(path_final_metrics, filename)
            df_holo_metrics = pd.read_csv(path_holo_metrics, index_col = 0)
            weakly_correlated_cols = ['ligand_iptm_mean', 'complex_plddt', 'complex_iplddt', 'iptm', 'ptm_boltz', 'ligand_iptm', 'rmsd_holo_binder_rfd_boltz',
                          'plddt', 'plddt_binder', 'binder_chain', 'target_chain', 'rmsd']
            drop_contact_cols = [x for x in df_holo_metrics.columns if ("paratope" in x) or ("epitope" in x)]
            full_drop_cols = weakly_correlated_cols + drop_contact_cols
            df_holo_copy = df_holo_metrics.drop(columns = full_drop_cols)
            df_holo_final = df_holo_copy.rename(columns = {'complex_plddt_mean' : 'plddt', 'complex_iplddt_mean' : 'iplddt', 'complex_pde_mean' : 'pde', 
                               'ptm_boltz_mean' : 'ptm_boltz', 'rmsd_holo_binder_rfd_boltz_mean' : 'rmsd_holo_binder_rfd_boltz',
                               'seq_binder' : 'sequence', 'seq' : 'seq_target_binder', 'protein_iptm_mean' : 'iptm'})
    
    # Merge across all 3 different validation steps
    df_metrics_pt1 = pd.merge(left = df_holo_final, right = df_apo_final, on = 'design_name', how = 'inner')
    df_metrics = pd.merge(left = df_metrics_pt1, right = df_physics_final, on = 'design_name', how = 'inner')

    return df_metrics
    
#----------Use if trying to merge metrics across multiple distinct ProteinHunter runs
def merge_metrics_across_runs(overarching_folder_paths: list) -> pd.DataFrame:
    """ Merges metrics across multiple distinct ProteinHunter Runs"""
    metrics_across_runs = []
    for overarching_folder_path in overarching_folder_paths:
        df_metrics = merge_metrics(overarching_folder_path)
        print("Number of Designs in Run: ", df_metrics.shape[0])
        print("Number of metrics: ", len(df_metrics.columns))
        print("-----------------------------------------------------------")
        metrics_across_runs.append(df_metrics)
    
    df_final = pd.concat(metrics_across_runs, axis = 0).reset_index(drop = True)
    if df_final['sequence'].is_unique == False:
        df_final = df_final.drop_duplicates(subset = ['sequence'], keep = 'first')
        print("Designed Sequences are not unique")
    print("Number of Designs in Combined Runs: ", df_final.shape[0])
    return df_final

path_design_campaign = "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular/"
overarching_folder_paths = [f"{path_design_campaign}/rfdiffusion"]
df_metrics = merge_metrics_across_runs(overarching_folder_paths)
df_metrics

In [0]:
agg_funcs = ['mean', 'median', 'std']
df_apo_agg = df_apo.groupby('design_name').agg({
    "ptm" : agg_funcs,
    "complex_plddt" : agg_funcs,
})

df_apo_agg.reset_index(inplace = True)
df_apo_agg.columns = [("_".join(col)).strip("_") for col in df_apo_agg.columns]
df_apo_agg = pd.merge(df_apo_agg, df_apo[['design_name', 'path_boltz_apo_structure']][df_apo['model_id'] == 0], on = 'design_name', how = 'left')
df_apo_final = df_apo_agg.rename(columns = {'ptm_mean': 'ptm_apo', 'complex_plddt_mean': 'plddt_apo'})
df_apo_final

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
fig, ax = plt.subplots()
data_col = 'plddt_apo'
sns.histplot(x = data_col, data = df_apo_final, ax = ax)
ax.set_xlabel(data_col);
ax.set_title(f"Distribution of {data_col}");

In [0]:
# More signal from complex_plddt_mean than plddt from AF2 (Signal = Spread in Designs' values for given metric)
weakly_correlated_cols = ['ligand_iptm_mean', 'complex_plddt', 'complex_iplddt', 'iptm', 'ptm_boltz', 'ligand_iptm', 'rmsd_holo_binder_rfd_boltz',
                          'plddt', 'plddt_binder', 'binder_chain', 'target_chain', 'rmsd']
drop_contact_cols = [x for x in df_holo.columns if ("paratope" in x) or ("epitope" in x)]
full_drop_cols = weakly_correlated_cols + drop_contact_cols
df_holo_copy = df_holo.drop(columns = full_drop_cols)
df_holo_final = df_holo_copy.rename(columns = {'complex_plddt_mean' : 'plddt', 'complex_iplddt_mean' : 'iplddt', 'complex_pde_mean' : 'pde', 
                               'ptm_boltz_mean' : 'ptm_boltz', 'rmsd_holo_binder_rfd_boltz_mean' : 'rmsd_holo_binder_rfd_boltz',
                               'seq_binder' : 'sequence', 'seq' : 'seq_target_binder', 'protein_iptm_mean' : 'iptm'})
print(df_holo_final.columns)

In [0]:
df_physics_final = df_physics.copy()
df_physics_final['design_name'] = df_physics_final['pdb_filename'].str.split('_model_').str[0]

In [0]:
df_metrics_pt1 = pd.merge(left = df_holo_final, right = df_apo_final, on = 'design_name', how = 'inner')
df_metrics = pd.merge(left = df_metrics_pt1, right = df_physics_final, on = 'design_name', how = 'inner')
df_metrics

In [0]:
df_physics.columns

In [0]:
fig, ax = plt.subplots()
data_col = 'surface_hydrophobicity'
sns.histplot(x = data_col, data = df_metrics, ax = ax)
ax.set_xlabel(data_col);
ax.set_title(f"Distribution of {data_col}");

In [0]:
print("Data Metric: ", data_col)
df_metrics[data_col].describe()

In [0]:
df_metrics.sort_values(data_col, ascending= True)

### Adding RMSD Metrics to Final Ranking

In [0]:
rmsd_threshold_weight = {
    'neg_rmsd_holo_binder_rfd_boltz' : {'threshold' : -2,  'weight' : 2},
    'neg_rmsd_holo_binder_rfd_af2' : {'threshold' : -2,  'weight' : 2},
    'neg_interface_holo_apo_rmsd' : {'threshold' : -1.5,  'weight' : 1.5}, 
    'neg_rmsd_motif' : {'threshold': 1.5, 'weight' : -1},
    'neg_rmsd_middle_strand' : {'threshold' : 1.5, 'weight' : 1},
}
rmsd_cols = [x for x in df_metrics.columns if (('rmsd' in x) & ('median' not in x) & ('std' not in x))]
rmsd_cols

In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/Scfv_Toolkit")
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics") 
from RankDesign import RankDesign
rank_design_obj = RankDesign()

for rmsd_col in rmsd_cols:
    neg_rmsd_col = f"neg_{rmsd_col}"
    threshold = rmsd_threshold_weight[neg_rmsd_col]['threshold']
    weight = rmsd_threshold_weight[neg_rmsd_col]['weight']

    # Add new metric to ranking scheme. Specifically, add the metric name, weight (importance in ranking), threshold for success, and whether to flip
    rank_design_obj.update_threshold(metric = neg_rmsd_col, threshold = threshold)
    rank_design_obj.update_weight(metric = neg_rmsd_col, weight = weight)
    rank_design_obj.update_cols_to_flip(metric = rmsd_col)

In [0]:
from StrucTools import *
from RFDUtils import *

list_sequence_designed = []
for index, row in df_metrics.iterrows():
    
    contig = df_metrics['contig'].iloc[index]
    filepath_apo = df_metrics['path_boltz_apo_structure'].iloc[index]
    sequence = df_metrics['sequence'].iloc[index]

    # 1. Extract Binder Contig
    print("Contig: ", contig)
    binder_contig = extract_binder_contig(contig)
    print("Binder_contigs: ", binder_contig)

    # 2. Create mask aligning to motif with True and non-motif or design = False
    example_coords, align_mask = extract_design_motif_coords(filepath_apo, binder_contig, binder_chain_id= 'A', 
                                                             return_coords_mask = True, desired_motif_chain_id= "")
    #3. Create design_mask
    design_mask = ~np.array(align_mask)
    sequence_designed = "".join([sequence[index] for index, val in enumerate(design_mask) if val == True])
    list_sequence_designed.append(sequence_designed)

df_metrics['sequence_designed'] = list_sequence_designed
df_metrics

### Run Entire Filter and Ranking Pipeline

In [0]:
top_n = 10
alpha = 0.1
df_top_designs = rank_design_obj.run_filter_rank_pipeline(df_designs = df_metrics, 
                                                          design_mask = design_mask, 
                                                          top_n = top_n,
                                                          alpha = alpha)
df_top_designs.reset_index(drop = True, inplace = True)
df_top_designs
                                                

In [0]:
print(df_top_designs['chains_ptm'].iloc[0])
df_top_designs['ptm_apo'].iloc[0]

In [0]:
list_sequence_motif = []
for index, row in df_top_designs.iterrows():
    
    contig = df_top_designs['contig'].iloc[index]
    filepath_apo = df_top_designs['path_boltz_apo_structure'].iloc[index]
    sequence = df_top_designs['sequence'].iloc[index]

    # 1. Extract Binder Contig
    print("Contig: ", contig)
    binder_contig = extract_binder_contig(contig)
    print("Binder_contigs: ", binder_contig)

    # 2. Create mask aligning to motif with True and non-motif or design = False
    example_coords, motif_mask = extract_design_motif_coords(filepath_apo, binder_contig, binder_chain_id= 'A', 
                                                             return_coords_mask = True, desired_motif_chain_id= "")
    #3. Create design_mask
    sequence_motif = "".join([sequence[index] for index, val in enumerate(motif_mask) if val == True])
    list_sequence_motif.append(sequence_motif)

assert(len(np.unique(list_sequence_motif)) == 1)

In [0]:
import biotite.sequence as seq
tag_seq = "MLEHHHHHH"
tag_obj = seq.ProteinSequence(sequence = tag_seq)
tag_weight = tag_obj.get_molecular_weight()
print("Tag Weight: ", tag_weight)
df_top_designs['mol_weight'] = df_top_designs['sequence'].apply(lambda sequence: seq.ProteinSequence(sequence = sequence).get_molecular_weight()) + tag_weight
df_top_designs['mol_weight']

In [0]:
df_top_designs.columns

In [0]:
for path in df_top_designs['path_holo_af2_structure'].values:
    visualize_structure(path)